# NV center ground-state physics

*A worked example for `spin_j.py`/`nv_center.py` -- new general spin-1 machinery, not an AMO.jl port.*

The nitrogen-vacancy center's ground state (3A2) is a spin-1 triplet ($m_s=-1,0,+1$), split at
zero field by $D \approx 2.87$ GHz -- something the package's existing spin-1/2-only `spin.py`
cannot represent at all. This notebook builds that structure with `spin_j`/`nv_center`, drives a
microwave transition, and demonstrates the simplified readout-contrast model in `isc_dephasing`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

import htdse as ht
from htdse.submodules.spin_j import spin_operators

D, E, g_e = 2.87e9, 5e6, 2.0028   # Hz -- commonly cited NV ground-state ZFS; E is illustrative strain
Sx, Sy, Sz = spin_operators(1.0)

## 1. Zero-field splitting

`zero_field_splitting(D, E)` builds $D(S_z^2 - S(S+1)/3) + E(S_x^2-S_y^2)$. At $E=0$ the $m_s=\pm1$
sublevels are exactly degenerate, split from $m_s=0$ by exactly $D$; turning on strain/electric-field
$E$ splits that degeneracy by exactly $2E$ (verified exactly in `tests/test_nv_center.py` -- this
isn't a numerical coincidence, $S_x^2-S_y^2$ only connects states two apart in $m$, and $m_s=0$ has no
such partner in a spin-1 manifold, so it stays an exact eigenstate).

In [ ]:
evals_D_only = np.linalg.eigvalsh(np.asarray(ht.zero_field_splitting(D, 0.0).hamiltonian(0)))
print("levels at E=0 (Hz):", evals_D_only)
print(f"ms=0 <-> ms=+-1 splitting: {evals_D_only[2] - evals_D_only[0]:.4e} Hz (== D)")

evals0 = np.linalg.eigvalsh(np.asarray(ht.zero_field_splitting(D, E).hamiltonian(0)))
print("levels with E={:.0e} (Hz):".format(E), evals0)
print(f"ms=+1 <-> ms=-1 splitting: {evals0[2] - evals0[1]:.4e} Hz (== 2E)")

## 2. Zeeman diagram

A field along the NV axis (`zeeman(g_e, Bz)`) shifts $m_s=+1$ up and $m_s=-1$ down linearly -- this
is the bias field real experiments apply to spectrally resolve the two microwave transitions from
each other (next section).

In [ ]:
Bzs = np.linspace(0, 2e7, 100)
levels = np.array([np.linalg.eigvalsh(np.asarray(
    (ht.zero_field_splitting(D, E) + ht.zeeman(g_e, Bz)).hamiltonian(0)))
    for Bz in Bzs])

fig, ax = plt.subplots(figsize=(5, 4))
for i in range(3):
    ax.plot(Bzs / 1e6, (levels[:, i] - evals0[0]) / 1e9)
ax.set_xlabel("Bz (MHz, arb. axial-field units)")
ax.set_ylabel("Energy above ms=0 (GHz)")
ax.set_title("NV ground-state Zeeman diagram")
plt.show()

## 3. Selective microwave drive (rotating-wave approximation)

$S_x$ couples $m_s=0$ to *both* $m_s=\pm1$ -- driving resonant with one transition means the other
sits at a large but finite detuning, not infinite. Rather than integrate a GHz-frequency drive
directly (numerically wasteful and not what any real control software does either), we move to the
frame rotating at the drive frequency: $m_s=0$ and the target ($m_s=-1$) become degenerate, and the
spectator ($m_s=+1$) sits at its own residual detuning $2 g_e B_z$ in that frame. A small bias field
already gives excellent selectivity; a larger one gives even less leakage.

In [ ]:
def rwa_rabi(Bz, Omega_mw, tmax, n=1000):
    E_p1, E_0, E_m1 = D / 3 + g_e * Bz, -2 * D / 3, D / 3 - g_e * Bz
    wd = E_m1 - E_0                    # resonant with ms=0 <-> ms=-1
    delta_p1 = (E_p1 - E_0) - wd         # ms=+1's residual detuning in this rotating frame
    H_rwa_diag = np.diag([delta_p1, 0.0, 0.0]).astype(complex)  # basis order: ms=+1,0,-1
    with ht.quiet():
        H = ht.term(H_rwa_diag, on="e") + ht.term(Sx, on="e", coeff=Omega_mw / 2)
        psi0 = np.array([0, 1, 0], dtype=complex)  # ms=0
        ts = np.linspace(0, tmax, n)
        ev = ht.HamiltonianEvolution(H, psi0, ladders=())  # e is a spin, not a truncated ladder
        pops = np.abs(ev.state_at(ts)) ** 2
    return ts, pops

Omega_mw = 3e5
fig, ax = plt.subplots(figsize=(6, 4))
for Bz, style in ((1e6, "--"), (5e6, "-")):
    ts, pops = rwa_rabi(Bz, Omega_mw, 2e-5)
    ax.plot(ts * 1e6, pops[:, 2], style, label=f"ms=-1, Bz={Bz/1e6:.0f} MHz")
    print(f"Bz={Bz:.0e}: leakage into ms=+1 = {pops[:,0].max():.5f}, "
          f"max transfer to ms=-1 = {pops[:,2].max():.5f}")
ax.set_xlabel("t (us)"); ax.set_ylabel("population")
ax.set_title("Selective ms=0 -> ms=-1 Rabi drive: larger bias field = cleaner addressing")
ax.legend()
plt.show()

## 4. Readout contrast: the `isc_dephasing` proxy

Real NV readout works because $m_s=\pm1$ shelve into a metastable singlet under illumination much
faster than $m_s=0$ does -- those extra orbital levels aren't modeled here. `isc_dephasing` is an
explicitly-labeled simplification: a pure-dephasing Lindblad jump per sublevel with its own rate,
reproducing the *coherence loss* that different shelving rates cause without carrying the singlet
levels through the simulation. Starting in an equal superposition of $m_s=0,-1$ with $m_s=-1$
dephasing ~30x faster (illustrative), the coherence between them decays:

In [ ]:
rho0 = np.array([[0, 0, 0], [0, 0.5, 0.5], [0, 0.5, 0.5]], dtype=complex)  # (ms=0, ms=-1) superposition
L = ht.isc_dephasing((0.0, 1e5, 3e6))  # (rate_ms=+1, rate_ms=0, rate_ms=-1)

with ht.quiet():
    ev = ht.LindbladEvolution(ht.zero_field_splitting(D, E) + L, rho0, ladders=())
    ts = np.linspace(0, 5e-6, 200)
    rhos = ev.state_at(ts)
coherence = np.abs(rhos[:, 1, 2])

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ts * 1e6, coherence)
ax.set_xlabel("t (us)"); ax.set_ylabel("|coherence(ms=0, ms=-1)|")
ax.set_title("Simplified readout-contrast decay (isc_dephasing)")
plt.show()